# SLV/GLD Pairs Trading Strategy

An out-of-sample mean-reversion backtest using SLV and GLD.

This notebook estimates the hedge ratio on a formation period, validates the spread with Engle–Granger and ADF tests, and trades the relationship on a separate period.

In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import shinybroker as sb
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint, adfuller
import plotly.express as px
import plotly.graph_objects as go

OUTPUT_DIR = Path.cwd() / "results"
OUTPUT_DIR.mkdir(exist_ok=True)

## 1. why SLV and GLD

i picked SLV (silver ETF) and GLD (gold ETF) because theyre both precious metals that react to the same stuff — inflation, interest rates, dollar strength, safe haven demand etc. silver historically moves with gold but with more volatility so theres a natural spread relationship between them. the gold-silver ratio has been a thing for literally centuries and tends to mean revert which is exactly what we need for pairs trading.

the main idea is that when silver gets too cheap or too expensive relative to gold, the ratio should eventually come back to normal. thats our edge.

In [4]:
slv = sb.Contract({
    "symbol": "SLV",
    "secType": "STK",
    "exchange": "SMART",
    "currency": "USD"
})

gld = sb.Contract({
    "symbol": "GLD",
    "secType": "STK",
    "exchange": "SMART",
    "currency": "USD"
})

slv_raw = sb.fetch_historical_data(
    contract=slv,
    barSizeSetting="1 day",
    durationStr="2 Y",
    whatToShow="ADJUSTED_LAST"
)["hst_dta"].copy()

gld_raw = sb.fetch_historical_data(
    contract=gld,
    barSizeSetting="1 day",
    durationStr="2 Y",
    whatToShow="ADJUSTED_LAST"
)["hst_dta"].copy()

slv_raw["date"] = pd.to_datetime(slv_raw["timestamp"])
gld_raw["date"] = pd.to_datetime(gld_raw["timestamp"])

prices = pd.merge(
    slv_raw[["date", "open", "close"]].rename(
        columns={"open": "slv_open", "close": "slv_close"}
    ),
    gld_raw[["date", "open", "close"]].rename(
        columns={"open": "gld_open", "close": "gld_close"}
    ),
    on="date",
    how="inner"
).sort_values("date").reset_index(drop=True)

print(f"Total trading days fetched: {len(prices)}")
print(prices.head())
print(prices.tail())

Total trading days fetched: 501
        date  slv_open  slv_close  gld_open  gld_close
0 2024-09-24     28.27      29.38    243.39     246.07
1 2024-09-25     29.10      29.06    246.19     245.73
2 2024-09-26     29.32      29.26    246.42     246.98
3 2024-09-27     29.34      28.86    246.31     245.02
4 2024-09-30     28.51      28.41    243.97     243.06
          date  slv_open  slv_close  gld_open  gld_close
496 2026-09-17     59.04      58.97    400.27     398.36
497 2026-09-18     60.08      59.93    400.48     401.17
498 2026-09-21     60.14      59.63    400.13     398.38
499 2026-09-22     59.31      60.73    397.07     400.07
500 2026-09-23     59.00      58.37    395.34     392.92


In [5]:
formation_start = "2024-06-01"
formation_end = "2025-05-31"
trading_start = "2025-06-01"
trading_end = "2026-03-25"

formation = prices[
    (prices["date"] >= formation_start) & (prices["date"] <= formation_end)
].copy().reset_index(drop=True)

trading = prices[
    (prices["date"] >= trading_start) & (prices["date"] <= trading_end)
].copy().reset_index(drop=True)

print(f"Formation period: {formation['date'].min().date()} to {formation['date'].max().date()}")
print(f"  Trading days: {len(formation)}")
print(f"Trading period:   {trading['date'].min().date()} to {trading['date'].max().date()}")
print(f"  Trading days: {len(trading)}")

Formation period: 2024-09-24 to 2025-05-30
  Trading days: 171
Trading period:   2025-06-02 to 2026-03-25
  Trading days: 205


## 2. statistical tests

before running the strategy i need to make sure this pair actually works statistically. checking correlation first as a quick screen, then cointegration (engle-granger), and then ADF on the spread to confirm its stationary.

In [6]:
corr = formation["slv_close"].corr(formation["gld_close"])

print("=" * 60)
print("CORRELATION ANALYSIS (Formation Period)")
print("=" * 60)
print(f"Pearson correlation (closing prices): {corr:.4f}")

fig = px.scatter(
    formation,
    x="gld_close",
    y="slv_close",
    trendline="ols",
    title="SLV vs GLD — Formation Period Closing Prices",
    labels={"gld_close": "GLD Close ($)", "slv_close": "SLV Close ($)"}
)
fig.update_layout(template="plotly_white", width=750, height=500)
fig.show()

CORRELATION ANALYSIS (Formation Period)
Pearson correlation (closing prices): 0.5815


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [ ]:
coint_stat, p_value, crit_values = coint(
    formation["slv_close"], formation["gld_close"]
)

print("=" * 60)
print("ENGLE-GRANGER COINTEGRATION TEST")
print("=" * 60)
print(f"Test statistic: {coint_stat:.4f}")
print(f"P-value:        {p_value:.4f}")
print(f"Critical values (1%, 5%, 10%): {crit_values}")
print()

if p_value < 0.05:
    print("REJECT the null of no cointegration at the 5% level.")
    print("the pair looks cointegrated — good to go for pairs trading.")
else:
    print("FAIL to reject the null of no cointegration at 5%.")
    print("proceeding with caution — the pair may not be well-cointegrated.")
    print("this is a known risk discussed in section 6.")

: 

In [ ]:
X_form = sm.add_constant(formation["gld_close"])
y_form = formation["slv_close"]
ols_model = sm.OLS(y_form, X_form).fit()

hedge_ratio = ols_model.params.iloc[1]
intercept = ols_model.params.iloc[0]

print("=" * 60)
print("OLS HEDGE RATIO (Formation Period)")
print("=" * 60)
print(f"SLV = {intercept:.4f} + {hedge_ratio:.6f} * GLD + e")
print(f"Hedge ratio (beta): {hedge_ratio:.6f}")
print(f"R-squared:          {ols_model.rsquared:.4f}")
print()
print(ols_model.summary())

formation["spread"] = formation["slv_close"] - hedge_ratio * formation["gld_close"]
spread_mean = formation["spread"].mean()
spread_std = formation["spread"].std()

print()
print(f"Formation spread mean: {spread_mean:.4f}")
print(f"Formation spread std:  {spread_std:.4f}")

: 

In [ ]:
window = 60
rolling_betas = []
for i in range(window, len(formation)):
    X_roll = sm.add_constant(formation["gld_close"].iloc[i - window:i])
    y_roll = formation["slv_close"].iloc[i - window:i]
    roll_model = sm.OLS(y_roll, X_roll).fit()
    rolling_betas.append({
        "date": formation["date"].iloc[i],
        "beta": roll_model.params.iloc[1]
    })

rolling_df = pd.DataFrame(rolling_betas)

fig = px.line(
    rolling_df, x="date", y="beta",
    title="Rolling 60-Day Hedge Ratio vs Static OLS Ratio"
)
fig.add_hline(
    y=hedge_ratio, line_dash="dash", line_color="red",
    annotation_text=f"Static beta = {hedge_ratio:.4f}"
)
fig.update_layout(template="plotly_white", yaxis_title="Hedge Ratio (beta)")
fig.show()

: 

In [ ]:
adf_result = adfuller(formation["spread"], autolag="AIC")

print("=" * 60)
print("AUGMENTED DICKEY-FULLER TEST ON SPREAD")
print("=" * 60)
print(f"ADF statistic: {adf_result[0]:.4f}")
print(f"P-value:       {adf_result[1]:.4f}")
print(f"Lags used:     {adf_result[2]}")
print(f"Observations:  {adf_result[3]}")
print()
print("Critical values:")
for key, val in adf_result[4].items():
    print(f"  {key}: {val:.4f}")
print()

if adf_result[1] < 0.05:
    print("REJECT the null of a unit root at 5%.")
    print("the spread looks stationary — mean-reversion should work.")
else:
    print("FAIL to reject the null of a unit root at 5%.")
    print("the spread may not be stationary — theres some risk here.")

: 

In [ ]:
formation["z_score"] = (formation["spread"] - spread_mean) / spread_std

fig = px.line(formation, x="date", y="z_score",
              title="Formation Period — Spread Z-Score")
fig.add_hline(y=2.0, line_dash="dash", line_color="red",
              annotation_text="Entry +2.0")
fig.add_hline(y=-2.0, line_dash="dash", line_color="green",
              annotation_text="Entry -2.0")
fig.add_hline(y=0.0, line_dash="dot", line_color="gray",
              annotation_text="Exit 0.0")
fig.add_hline(y=3.0, line_dash="dash", line_color="darkred",
              annotation_text="Stop +3.0")
fig.add_hline(y=-3.0, line_dash="dash", line_color="darkgreen",
              annotation_text="Stop -3.0")
fig.update_layout(template="plotly_white", yaxis_title="Z-Score",
                  xaxis_title="Date")
fig.show()

: 

## 3. trading rules

using a static OLS hedge ratio from the formation period. looked at the rolling OLS plot above and the 60-day beta does move around which means a kalman filter or rolling approach could adapt better, but for now keeping it simple with the static ratio to avoid overfitting.

entry: short the spread when z-score goes above +2.0, long the spread when it drops below -2.0

exit: close when z-score crosses back to 0 (mean reversion). if the spread blows out past +/-3.0 thats a stop loss. also closing out after 20 trading days if nothing happens. any position still open at the end of the backtest gets closed at the last close price.

position size is 30% of NAV per trade split across both legs. signals come from the previous days close, trades happen at the next open so theres no look-ahead.

In [ ]:
entry_threshold = 2.0
exit_threshold = 0.0
stop_loss_threshold = 3.0
max_hold_days = 20
initial_capital = 1_000_000.0
position_pct = 0.30

print("=" * 60)
print("STRATEGY PARAMETERS")
print("=" * 60)
print(f"Entry Z-score:       +/-{entry_threshold}")
print(f"Exit Z-score:        {exit_threshold}")
print(f"Stop-loss Z-score:   +/-{stop_loss_threshold}")
print(f"Max hold period:     {max_hold_days} trading days")
print(f"Initial capital:     ${initial_capital:,.0f}")
print(f"Position size:       {position_pct:.0%} of NAV")

: 

## 4. backtest

running the strategy on the trading period (jun 2025 to mar 2026). hedge ratio and spread stats come from the formation period only so theres no data snooping.

In [ ]:
trading = trading.copy()
trading["spread"] = trading["slv_close"] - hedge_ratio * trading["gld_close"]
trading["z_score"] = (trading["spread"] - spread_mean) / spread_std
trading["signal_z"] = trading["z_score"].shift(1)

trades = []
ledger_rows = []
cash = initial_capital
slv_pos = 0
gld_pos = 0
in_trade = False
trade_entry = {}
hold_days = 0

for idx in range(len(trading)):
    row = trading.iloc[idx]
    date = row["date"]
    slv_open, gld_open = row["slv_open"], row["gld_open"]
    slv_close, gld_close = row["slv_close"], row["gld_close"]
    signal_z = row["signal_z"]
    exited_today = False

    if pd.isna(signal_z):
        ledger_rows.append({
            "date": date, "slv_position": 0, "gld_position": 0,
            "slv_close": slv_close, "gld_close": gld_close,
            "mkt_val": 0.0, "cash": cash, "NAV": cash
        })
        continue

    if in_trade:
        hold_days += 1
        exit_reason = None
        direction = trade_entry["direction"]

        if direction == "short_spread":
            if signal_z <= exit_threshold:
                exit_reason = "mean_reversion"
            elif signal_z >= stop_loss_threshold:
                exit_reason = "stop_loss"
        else:
            if signal_z >= exit_threshold:
                exit_reason = "mean_reversion"
            elif signal_z <= -stop_loss_threshold:
                exit_reason = "stop_loss"

        if hold_days >= max_hold_days and exit_reason is None:
            exit_reason = "time_limit"

        if exit_reason:
            cash += slv_pos * slv_open + gld_pos * gld_open

            pnl = (
                slv_pos * (slv_open - trade_entry["slv_entry_price"]) +
                gld_pos * (gld_open - trade_entry["gld_entry_price"])
            )

            trades.append({
                "entry_date": trade_entry["entry_date"],
                "exit_date": date,
                "direction": direction,
                "slv_qty": trade_entry["slv_qty"],
                "gld_qty": trade_entry["gld_qty"],
                "slv_entry_price": trade_entry["slv_entry_price"],
                "slv_exit_price": slv_open,
                "gld_entry_price": trade_entry["gld_entry_price"],
                "gld_exit_price": gld_open,
                "pnl": round(pnl, 2),
                "hold_days": hold_days,
                "exit_reason": exit_reason,
                "entry_z": trade_entry["entry_z"],
                "exit_z": signal_z
            })

            slv_pos = 0
            gld_pos = 0
            in_trade = False
            hold_days = 0
            exited_today = True

    if not in_trade and not exited_today:
        if signal_z > entry_threshold or signal_z < -entry_threshold:
            nav = cash
            trade_notional = position_pct * nav
            n_shares = int(trade_notional / (slv_open + abs(hedge_ratio) * gld_open))
            m_shares = max(1, int(abs(hedge_ratio) * n_shares))

            if signal_z > entry_threshold:
                slv_pos = -n_shares
                gld_pos = m_shares
                direction = "short_spread"
            else:
                slv_pos = n_shares
                gld_pos = -m_shares
                direction = "long_spread"

            cash -= slv_pos * slv_open + gld_pos * gld_open

            in_trade = True
            trade_entry = {
                "entry_date": date,
                "direction": direction,
                "slv_qty": slv_pos,
                "gld_qty": gld_pos,
                "slv_entry_price": slv_open,
                "gld_entry_price": gld_open,
                "entry_z": signal_z
            }
            hold_days = 0

    mkt_val = slv_pos * slv_close + gld_pos * gld_close
    nav = cash + mkt_val
    ledger_rows.append({
        "date": date, "slv_position": slv_pos, "gld_position": gld_pos,
        "slv_close": slv_close, "gld_close": gld_close,
        "mkt_val": mkt_val, "cash": cash, "NAV": nav
    })

if in_trade:
    last = trading.iloc[-1]
    pnl = (
        slv_pos * (last["slv_close"] - trade_entry["slv_entry_price"]) +
        gld_pos * (last["gld_close"] - trade_entry["gld_entry_price"])
    )
    trades.append({
        "entry_date": trade_entry["entry_date"],
        "exit_date": last["date"],
        "direction": trade_entry["direction"],
        "slv_qty": trade_entry["slv_qty"],
        "gld_qty": trade_entry["gld_qty"],
        "slv_entry_price": trade_entry["slv_entry_price"],
        "slv_exit_price": last["slv_close"],
        "gld_entry_price": trade_entry["gld_entry_price"],
        "gld_exit_price": last["gld_close"],
        "pnl": round(pnl, 2),
        "hold_days": hold_days,
        "exit_reason": "end_of_backtest",
        "entry_z": trade_entry["entry_z"],
        "exit_z": last["z_score"]
    })

blotter = pd.DataFrame(trades)
ledger = pd.DataFrame(ledger_rows)

print(f"Total trades executed: {len(blotter)}")
print()

if len(blotter) > 0:
    print("=" * 70)
    print("MATCHED BLOTTER (trades)")
    print("=" * 70)
    print(blotter.to_string(index=False))

print()
print("=" * 70)
print("LEDGER (last 10 rows)")
print("=" * 70)
print(ledger.tail(10).to_string(index=False))

: 

In [ ]:
blotter.to_csv(OUTPUT_DIR / "trades.csv", index=False)
ledger.to_csv(OUTPUT_DIR / "ledger.csv", index=False)

print(f"trades.csv saved to {OUTPUT_DIR}")
print(f"ledger.csv saved to {OUTPUT_DIR}")
print(f"Blotter rows: {len(blotter)}")
print(f"Ledger rows:  {len(ledger)}")

: 

## 5. results

In [ ]:
ledger["daily_return"] = ledger["NAV"].pct_change()

total_return = (ledger["NAV"].iloc[-1] / ledger["NAV"].iloc[0]) - 1
n_days = len(ledger)
ann_return = (1 + total_return) ** (252 / n_days) - 1
ann_vol = ledger["daily_return"].std() * np.sqrt(252)

risk_free_rate = 0.0375
sharpe = (ann_return - risk_free_rate) / ann_vol if ann_vol > 0 else np.nan

ledger["peak"] = ledger["NAV"].cummax()
ledger["drawdown"] = (ledger["NAV"] - ledger["peak"]) / ledger["peak"]
max_drawdown = ledger["drawdown"].min()

if len(blotter) > 0:
    win_rate = (blotter["pnl"] > 0).mean()
    avg_hold = blotter["hold_days"].mean()
    avg_pnl = blotter["pnl"].mean()
    total_pnl = blotter["pnl"].sum()
    n_trades = len(blotter)

    cost_per_share = 0.005
    total_shares_traded = blotter.apply(
        lambda r: abs(r["slv_qty"]) + abs(r["gld_qty"]), axis=1
    ).sum() * 2
    estimated_costs = total_shares_traded * cost_per_share
else:
    win_rate = avg_hold = avg_pnl = total_pnl = n_trades = 0
    estimated_costs = 0.0

print("=" * 60)
print("PERFORMANCE SUMMARY")
print("=" * 60)
print(f"Total return:            {total_return:.4%}")
print(f"Annualized return:       {ann_return:.4%}")
print(f"Annualized volatility:   {ann_vol:.4%}")
print(f"Sharpe ratio (rf=3.75%): {sharpe:.4f}")
print(f"Max drawdown:            {max_drawdown:.4%}")
print(f"Number of trades:        {n_trades}")
print(f"Win rate:                {win_rate:.2%}")
print(f"Avg hold time (days):    {avg_hold:.1f}")
print(f"Avg PnL per trade:       ${avg_pnl:,.2f}")
print(f"Total PnL:               ${total_pnl:,.2f}")
print(f"Est. transaction costs:  ${estimated_costs:,.2f}")
print(f"Net PnL after costs:     ${total_pnl - estimated_costs:,.2f}")

: 

In [ ]:
fig = px.line(
    ledger, x="date", y="NAV",
    title="Pairs Trading Strategy — Equity Curve (SLV / GLD)"
)
fig.update_layout(
    template="plotly_white",
    yaxis_title="NAV ($)",
    xaxis_title="Date",
    width=900, height=500
)
fig.show()

: 

In [ ]:
fig = px.area(
    ledger, x="date", y="drawdown",
    title="Strategy Drawdowns"
)
fig.update_layout(
    template="plotly_white",
    yaxis_title="Drawdown",
    xaxis_title="Date",
    yaxis_tickformat=".2%",
    width=900, height=400
)
fig.show()

: 

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=trading["date"], y=trading["z_score"],
    mode="lines", name="Z-Score",
    line=dict(color="steelblue", width=1.5)
))

fig.add_hline(y=entry_threshold, line_dash="dash", line_color="red",
              annotation_text=f"+{entry_threshold}")
fig.add_hline(y=-entry_threshold, line_dash="dash", line_color="green",
              annotation_text=f"-{entry_threshold}")
fig.add_hline(y=0, line_dash="dot", line_color="gray")
fig.add_hline(y=stop_loss_threshold, line_dash="dash", line_color="darkred",
              annotation_text=f"+{stop_loss_threshold} (stop)")
fig.add_hline(y=-stop_loss_threshold, line_dash="dash", line_color="darkgreen",
              annotation_text=f"-{stop_loss_threshold} (stop)")

if len(blotter) > 0:
    for _, t in blotter.iterrows():
        entry_color = "red" if t["direction"] == "short_spread" else "green"
        exit_color = "orange" if t["exit_reason"] == "stop_loss" else "blue"
        if t["exit_reason"] == "time_limit":
            exit_color = "purple"

        fig.add_trace(go.Scatter(
            x=[t["entry_date"]], y=[t["entry_z"]],
            mode="markers",
            marker=dict(symbol="triangle-up", size=12, color=entry_color),
            showlegend=False
        ))
        fig.add_trace(go.Scatter(
            x=[t["exit_date"]], y=[t["exit_z"]],
            mode="markers",
            marker=dict(symbol="triangle-down", size=12, color=exit_color),
            showlegend=False
        ))

fig.update_layout(
    title="Trading Period Z-Score with Entry/Exit Markers",
    template="plotly_white",
    xaxis_title="Date", yaxis_title="Z-Score",
    width=900, height=500
)
fig.show()

print("Marker legend:")
print("  Green  triangle-up   = Long spread entry")
print("  Red    triangle-up   = Short spread entry")
print("  Blue   triangle-down = Mean-reversion exit")
print("  Orange triangle-down = Stop-loss exit")
print("  Purple triangle-down = Time-limit exit")

: 

In [ ]:
if len(blotter) > 0:
    fig = px.histogram(
        blotter, x="pnl", nbins=20,
        title="Distribution of Trade PnL",
        color_discrete_sequence=["steelblue"]
    )
    fig.add_vline(x=0, line_dash="dash", line_color="red")
    fig.update_layout(
        template="plotly_white",
        xaxis_title="PnL ($)", yaxis_title="Count",
        width=750, height=400
    )
    fig.show()

    print("=" * 60)
    print("EXIT REASON BREAKDOWN")
    print("=" * 60)
    for reason, count in blotter["exit_reason"].value_counts().items():
        subset = blotter[blotter["exit_reason"] == reason]
        avg = subset["pnl"].mean()
        print(f"  {reason:20s}: {count:3d} trades, avg PnL = ${avg:,.2f}")
else:
    print("No trades were executed during the trading period.")
    print("The Z-score may not have reached the +/-2.0 entry threshold.")
    print("Consider relaxing the entry threshold or choosing a different pair.")

: 

## 6. thoughts

**what works:**
- SLV and GLD are a solid pair with real economic logic behind them, both react to the same macro factors
- formation and trading periods dont overlap so no data snooping happening here
- the stop loss at +/-3.0 and 20 day time limit keep us from getting stuck in a bad trade forever
- using previous days close for signals and next days open for execution so no look-ahead bias

**risks:**
- cointegration can break down, especially during big macro events where gold and silver decouple temporarily
- the hedge ratio is fixed from the formation period so if the relationship shifts during trading were stuck with a stale beta. a kalman filter would fix this but adds complexity
- silver is less liquid than gold so larger orders might get worse fills
- transaction costs eat into PnL, our $0.005/share estimate doesnt include spread or market impact
- only trading one pair so all the risk is concentrated

**what id do differently:**
1. kalman filter for the hedge ratio so it updates dynamically
2. trade multiple precious metal pairs to diversify (SLV/GLD, SLV/PPLT, GLD/PPLT etc)
3. expand beyond precious metals and look at pairs in oil (like USO/XLE) or currency ETFs to diversify across multiple sectors instead of concentrating in one pair. especially when trump is president with all these wars.
3. adaptive z-score thresholds that adjust with recent volatility instead of fixed +/-2.0
4. rolling cointegration check every 60 days, pause trading if the p-value gets too high
5. scale position size based on spread volatility so risk per trade stays more consistent